# LogiTrans Logistics: Object-Oriented Fleet Management Prototype
**Student:** Prajwol Shrestha | **Course:** AIM 401 – Python Programming for Data Science | **Professor:** Dhiraj Bashyal

This notebook builds a small fleet management system that uses all four pillars of OOP:
**abstraction**, **encapsulation**, **inheritance** and **polymorphism**.
Run all cells from top to bottom (*Run All*). The last cell runs `test_fleet.py`, so keep that file in the same folder as this notebook.

## Task 1 and Task 2: Abstract base class `Vehicle` with a protected daily rate

In [1]:
from abc import ABC, abstractmethod


class Vehicle(ABC):
    """Abstract blueprint for every rentable vehicle."""

    def __init__(self, vehicle_id, model, daily_rate):
        self.vehicle_id = vehicle_id
        self.model = model
        self.__daily_rate = None          # private (name-mangled) attribute
        self.set_daily_rate(daily_rate)   # reuse the validation logic

    # ---- Encapsulation: controlled access to the private rate ----
    def get_daily_rate(self):
        return self.__daily_rate

    def set_daily_rate(self, rate):
        if isinstance(rate, bool) or not isinstance(rate, (int, float)):
            raise ValueError("Daily rate must be a number")
        if rate <= 0.0:
            raise ValueError("Daily rate must be greater than 0.0")
        self.__daily_rate = float(rate)

    # ---- Abstraction: every subclass must supply its own billing ----
    @abstractmethod
    def calculate_rental_cost(self, days):
        """Return the total rental cost for the given number of days."""

    @staticmethod
    def _check_days(days):
        """Shared guard used by the subclasses' billing methods."""
        if isinstance(days, bool) or not isinstance(days, (int, float)) or days < 0:
            raise ValueError("Rental days must be a number that is 0 or more")

    # ---- Operator overloading: sort vehicles by daily rate ----
    def __lt__(self, other):
        if not isinstance(other, Vehicle):
            return NotImplemented
        return self.get_daily_rate() < other.get_daily_rate()

### Task 1 check: `Vehicle` cannot be instantiated

In [2]:
try:
    Vehicle("X-1", "Generic", 50.0)
except TypeError as error:
    print("TypeError:", error)

TypeError: Can't instantiate abstract class Vehicle without an implementation for abstract method 'calculate_rental_cost'


### Task 2 check: getter, setter, validation and the private attribute

In [3]:
class Probe(Vehicle):                       # tiny concrete class, only for this demo
    def calculate_rental_cost(self, days):
        return 0.0

p = Probe("P-1", "Probe", 40.0)
print("Initial rate      :", p.get_daily_rate())
p.set_daily_rate(55.5)
print("After set to 55.5 :", p.get_daily_rate())

for bad in (0, -20, "abc"):
    try:
        p.set_daily_rate(bad)
    except ValueError as error:
        print(f"set_daily_rate({bad!r:>6}) -> ValueError: {error}")

print("Rate after failed sets:", p.get_daily_rate())

try:
    print(p.__daily_rate)
except AttributeError as error:
    print("AttributeError:", error)
print("Mangled name exists:", hasattr(p, "_Vehicle__daily_rate"))

Initial rate      : 40.0
After set to 55.5 : 55.5
set_daily_rate(     0) -> ValueError: Daily rate must be greater than 0.0
set_daily_rate(   -20) -> ValueError: Daily rate must be greater than 0.0
set_daily_rate( 'abc') -> ValueError: Daily rate must be a number
Rate after failed sets: 55.5
AttributeError: 'Probe' object has no attribute '__daily_rate'
Mangled name exists: True


## Task 3: Child classes and Task 4: method overriding and operator overloading

In [4]:
class PassengerCar(Vehicle):
    SEAT_FEE = 15.0

    def __init__(self, vehicle_id, model, daily_rate, num_seats):
        super().__init__(vehicle_id, model, daily_rate)
        self.num_seats = num_seats

    def calculate_rental_cost(self, days):
        self._check_days(days)
        return (self.get_daily_rate() * days) + (self.SEAT_FEE * self.num_seats)

    def __str__(self):
        return (f"PassengerCar [{self.vehicle_id}] {self.model} | "
                f"{self.num_seats} seats | ${self.get_daily_rate():.2f}/day")


class CargoTruck(Vehicle):
    TONNAGE_FEE = 50.0

    def __init__(self, vehicle_id, model, daily_rate, cargo_capacity_tons):
        super().__init__(vehicle_id, model, daily_rate)
        self.cargo_capacity_tons = cargo_capacity_tons

    def calculate_rental_cost(self, days):
        self._check_days(days)
        return (self.get_daily_rate() * days) + (self.TONNAGE_FEE * self.cargo_capacity_tons)

    def __str__(self):
        return (f"CargoTruck  [{self.vehicle_id}] {self.model} | "
                f"{self.cargo_capacity_tons} tons | ${self.get_daily_rate():.2f}/day")

### Task 3 check: inheritance with `super().__init__()`

In [5]:
car = PassengerCar("PC-101", "Toyota Camry", 60.0, 5)
truck = CargoTruck("CT-201", "Volvo FH16", 150.0, 12.5)

print("car   attributes:", {k: v for k, v in vars(car).items() if not k.startswith("_")})
print("truck attributes:", {k: v for k, v in vars(truck).items() if not k.startswith("_")})
print("car   rate via getter:", car.get_daily_rate())
print("PassengerCar MRO:", [c.__name__ for c in PassengerCar.__mro__])
print("CargoTruck   MRO:", [c.__name__ for c in CargoTruck.__mro__])
print("issubclass(PassengerCar, Vehicle):", issubclass(PassengerCar, Vehicle))
print("issubclass(CargoTruck, Vehicle)  :", issubclass(CargoTruck, Vehicle))

car   attributes: {'vehicle_id': 'PC-101', 'model': 'Toyota Camry', 'num_seats': 5}
truck attributes: {'vehicle_id': 'CT-201', 'model': 'Volvo FH16', 'cargo_capacity_tons': 12.5}
car   rate via getter: 60.0
PassengerCar MRO: ['PassengerCar', 'Vehicle', 'ABC', 'object']
CargoTruck   MRO: ['CargoTruck', 'Vehicle', 'ABC', 'object']
issubclass(PassengerCar, Vehicle): True
issubclass(CargoTruck, Vehicle)  : True


### Task 4 check: the same method name, two different formulas

In [6]:
days = 3
print(f"{car.model:<14}: 60.0 x {days} + 15.0 x 5    =", car.calculate_rental_cost(days))
print(f"{truck.model:<14}: 150.0 x {days} + 50.0 x 12.5 =", truck.calculate_rental_cost(days))

Toyota Camry  : 60.0 x 3 + 15.0 x 5    = 255.0
Volvo FH16    : 150.0 x 3 + 50.0 x 12.5 = 1075.0


### Task 4 check: `__str__()` and `__lt__()`

In [7]:
print(car)
print(truck)
print()
print("car < truck :", car < truck)
print("truck < car :", truck < car)

fleet = [
    PassengerCar("PC-101", "Toyota Camry", 60.0, 5),
    CargoTruck("CT-201", "Volvo FH16", 150.0, 12.5),
    PassengerCar("PC-102", "Honda Odyssey", 85.0, 7),
    CargoTruck("CT-202", "Isuzu NPR", 110.0, 4.0),
]
print("\nSorted by daily rate:")
for v in sorted(fleet):
    print("  ", v)

PassengerCar [PC-101] Toyota Camry | 5 seats | $60.00/day
CargoTruck  [CT-201] Volvo FH16 | 12.5 tons | $150.00/day

car < truck : True
truck < car : False

Sorted by daily rate:
   PassengerCar [PC-101] Toyota Camry | 5 seats | $60.00/day
   PassengerCar [PC-102] Honda Odyssey | 7 seats | $85.00/day
   CargoTruck  [CT-202] Isuzu NPR | 4.0 tons | $110.00/day
   CargoTruck  [CT-201] Volvo FH16 | 12.5 tons | $150.00/day


## Task 4: Polymorphic processing with `generate_fleet_manifest()`

In [8]:
def generate_fleet_manifest(fleet_list, days):
    """Print a manifest for a mixed fleet and return the fleet total.

    The function never checks what kind of vehicle it is handling: each
    object decides its own billing through method overriding.
    """
    print(f"===== LogiTrans Fleet Manifest ({days} days) =====")
    total = 0.0
    for vehicle in fleet_list:
        cost = vehicle.calculate_rental_cost(days)   # polymorphic call
        print(f"{str(vehicle)} -> Total: ${cost:,.2f}")
        total += cost
    print(f"Fleet total: ${total:,.2f}")
    return total

In [9]:
fleet_total = generate_fleet_manifest(fleet, 3)

===== LogiTrans Fleet Manifest (3 days) =====
PassengerCar [PC-101] Toyota Camry | 5 seats | $60.00/day -> Total: $255.00
CargoTruck  [CT-201] Volvo FH16 | 12.5 tons | $150.00/day -> Total: $1,075.00
PassengerCar [PC-102] Honda Odyssey | 7 seats | $85.00/day -> Total: $360.00
CargoTruck  [CT-202] Isuzu NPR | 4.0 tons | $110.00/day -> Total: $530.00
Fleet total: $2,220.00


### Extensibility check: a new vehicle type works without changing the manifest function

In [10]:
class Van(Vehicle):
    def calculate_rental_cost(self, days):
        return self.get_daily_rate() * days + 25.0

    def __str__(self):
        return f"Van         [{self.vehicle_id}] {self.model} | ${self.get_daily_rate():.2f}/day"


_ = generate_fleet_manifest(fleet + [Van("V-301", "Ford Transit", 80.0)], 3)

===== LogiTrans Fleet Manifest (3 days) =====
PassengerCar [PC-101] Toyota Camry | 5 seats | $60.00/day -> Total: $255.00
CargoTruck  [CT-201] Volvo FH16 | 12.5 tons | $150.00/day -> Total: $1,075.00
PassengerCar [PC-102] Honda Odyssey | 7 seats | $85.00/day -> Total: $360.00
CargoTruck  [CT-202] Isuzu NPR | 4.0 tons | $110.00/day -> Total: $530.00
Van         [V-301] Ford Transit | $80.00/day -> Total: $265.00
Fleet total: $2,485.00


## Additional edge-case tests

In [11]:
def attempt(label, func):
    try:
        print(f"{label:<44} -> {func()}")
    except Exception as error:
        print(f"{label:<44} -> {type(error).__name__}: {error}")

attempt("car.calculate_rental_cost(0)", lambda: car.calculate_rental_cost(0))
attempt("car.calculate_rental_cost(1)", lambda: car.calculate_rental_cost(1))
attempt("truck.calculate_rental_cost(2.5)", lambda: truck.calculate_rental_cost(2.5))
attempt("truck.calculate_rental_cost(-2)", lambda: truck.calculate_rental_cost(-2))
attempt("PassengerCar('X','Mini',45.0,0) cost, 2 days", lambda: PassengerCar('X','Mini',45.0,0).calculate_rental_cost(2))
attempt("CargoTruck('X','Empty',90.0,0.0) cost, 2 days", lambda: CargoTruck('X','Empty',90.0,0.0).calculate_rental_cost(2))
attempt("PassengerCar('X','Bad',-1,4)", lambda: PassengerCar('X','Bad',-1,4))
attempt("CargoTruck('X','Bad','cheap',4.0)", lambda: CargoTruck('X','Bad','cheap',4.0))
attempt("generate_fleet_manifest([], 5) return value", lambda: generate_fleet_manifest([], 5))
attempt("sorted([]) on an empty fleet", lambda: sorted([]))
attempt("car < 100", lambda: car < 100)

car.calculate_rental_cost(0)                 -> 75.0
car.calculate_rental_cost(1)                 -> 135.0
truck.calculate_rental_cost(2.5)             -> 1000.0
truck.calculate_rental_cost(-2)              -> ValueError: Rental days must be a number that is 0 or more
PassengerCar('X','Mini',45.0,0) cost, 2 days -> 90.0
CargoTruck('X','Empty',90.0,0.0) cost, 2 days -> 180.0
PassengerCar('X','Bad',-1,4)                 -> ValueError: Daily rate must be greater than 0.0
CargoTruck('X','Bad','cheap',4.0)            -> ValueError: Daily rate must be a number
===== LogiTrans Fleet Manifest (5 days) =====
Fleet total: $0.00
generate_fleet_manifest([], 5) return value  -> 0.0
sorted([]) on an empty fleet                 -> []
car < 100                                    -> TypeError: '<' not supported between instances of 'PassengerCar' and 'int'


## Automated unit tests
Runs the 23 tests in `test_fleet.py` (needs `fleet.py` and `test_fleet.py` in the same folder).

In [12]:
!python test_fleet.py

test_subclass_without_billing_cannot_be_instantiated (__main__.TestAbstraction.test_subclass_without_billing_cannot_be_instantiated) ... ok
test_vehicle_cannot_be_instantiated (__main__.TestAbstraction.test_vehicle_cannot_be_instantiated) ... ok
test_failed_set_keeps_old_rate (__main__.TestEncapsulation.test_failed_set_keeps_old_rate) ... ok
test_getter_returns_rate (__main__.TestEncapsulation.test_getter_returns_rate) ... ok
test_invalid_rate_in_constructor (__main__.TestEncapsulation.test_invalid_rate_in_constructor) ... ok
test_negative_rate_rejected (__main__.TestEncapsulation.test_negative_rate_rejected) ... ok
test_non_numeric_rate_rejected (__main__.TestEncapsulation.test_non_numeric_rate_rejected) ... ok
test_private_attribute_hidden (__main__.TestEncapsulation.test_private_attribute_hidden) ... ok
test_setter_updates_rate (__main__.TestEncapsulation.test_setter_updates_rate) ... ok
test_zero_rate_rejected (__main__.TestEncapsulation.test_zero_rate_rejected) ... ok
test_base_at